In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer

In [2]:
import sys
sys.path.append("./cluster_anlys")

from kondrak_cluster_morphology import (
    run_kondrak_morphology_for_partition,   
    run_kondrak_morphology_for_random_pca_partition,
    concat_global_summary_rows,
    load_random_pca_summary_df,
)

In [3]:
seed_list = [0, 42, 1000, 9999, 1813382118, 827307999, 1627694678, 1911784257]

# Mistral-7B

In [14]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "mistralai/Mistral-7B-v0.1"
space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [5, 142, 997, 2084, 3031, 3459, 3938, 4088, 4096]

[✓] Tokenizer loaded from: mistralai/Mistral-7B-v0.1/tokenizer


In [5]:
summary_df = run_random_pca_hdbscan_grid(
    embedding_matrix=embedding_matrix,
    candidate_dims=candidate_dims,
    hdbscan_param_grid=hdbscan_param_grid,
    model_name=model_name,
    space_name=space_name,
    pca_seed=42,
    randomized_fit_dim=full_pca_dim,
    out_root="comp",
    l2_norm=True,
)
summary_df.head()

KeyboardInterrupt: 

In [6]:
# Collect global statistics for each PCA dimension
global_rows = []

for pca_dim in pca_dim_list:
    print(f"\n=== Running pca_dim={pca_dim} ===")

    # Construct output path for cluster-level CSV
    cluster_outpath = (
        f"{out_root}/{model_name}/{space_name}/morph/"
        f"kondrak_clusters_pca_{pca_dim}.csv"
    )

    Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)
    # Run morphology analysis for the current partition
    # Returns:
    #   - cluster_df: per-cluster metrics
    #   - global_row: aggregated statistics for this PCA dimension
    cluster_df, global_row = run_kondrak_morphology_for_partition(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_dim=pca_dim,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        token_col=token_col,
        cluster_id_col=cluster_id_col,
        ddof=0,
        print_columns=False,
        save_cluster_csv=True,               # enable saving cluster-level CSV
        tokenizer=tokenizer,
        cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
    )

    # Store global summary for this PCA dimension
    global_rows.append(global_row)

# Concatenate all global summaries into a single DataFrame
global_summary_df = concat_global_summary_rows(global_rows)

# Sort by PCA dimension for consistent ordering
global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

# Save global summary CSV
global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/kondrak_global_summary.csv",
    index=False
)


=== Running pca_dim=5 ===

=== Running pca_dim=142 ===

=== Running pca_dim=997 ===

=== Running pca_dim=2084 ===

=== Running pca_dim=3031 ===

=== Running pca_dim=3459 ===

=== Running pca_dim=3938 ===

=== Running pca_dim=4088 ===

=== Running pca_dim=4096 ===


In [ ]:
# =========================================================
# sum
# =========================================================
global_summary_df = concat_global_summary_rows(global_rows)

global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

global_summary_df

In [15]:
from pathlib import Path

# Collect global statistics for each seed and PCA dimension
all_seed_global_summary = []

for pca_seed in seed_list:
    print(f"\n==============================")
    print(f"=== Running pca_seed={pca_seed} ===")
    print(f"==============================")

    global_rows = []

    for pca_dim in pca_dim_list:
        print(f"\n=== Running pca_seed={pca_seed}, pca_dim={pca_dim} ===")

        # Construct output path for cluster-level CSV
        cluster_outpath = (
            f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/morph/"
            f"kondrak_clusters_pca_{pca_dim}.csv"
        )

        Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)

        # Run morphology analysis for the current random-PCA partition
        # Returns:
        #   - cluster_df: per-cluster metrics
        #   - global_row: aggregated statistics for this PCA dimension
        cluster_df, global_row = run_kondrak_morphology_for_random_pca_partition(
            out_root=out_root,
            model_name=model_name,
            space_name=space_name,
            pca_seed=pca_seed,
            pca_dim=pca_dim,
            l2_norm=l2_norm,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=metric,
            cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon,
            token_col=token_col,
            cluster_id_col=cluster_id_col,
            ddof=0,
            print_columns=False,
            save_cluster_csv=True,               # enable saving cluster-level CSV
            tokenizer=tokenizer,
            cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
        )

        # Store global summary for this PCA dimension
        global_rows.append(global_row)

    # Concatenate all global summaries for this seed into a single DataFrame
    global_summary_df = concat_global_summary_rows(global_rows)

    # Sort by PCA dimension for consistent ordering
    global_summary_df.sort_values("pca_dim", inplace=True)
    global_summary_df.reset_index(drop=True, inplace=True)

    # Save per-seed global summary CSV
    seed_summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/"
        f"kondrak_global_summary.csv"
    )
    Path(seed_summary_outpath).parent.mkdir(parents=True, exist_ok=True)
    global_summary_df.to_csv(seed_summary_outpath, index=False)

    all_seed_global_summary.append(global_summary_df)

# Optional: concatenate all seeds into one summary DataFrame
all_seed_global_summary_df = concat_global_summary_rows(all_seed_global_summary)

# Sort for consistent ordering
all_seed_global_summary_df.sort_values(
    ["pca_seed", "pca_dim"], inplace=True
)
all_seed_global_summary_df.reset_index(drop=True, inplace=True)

# Save all-seed summary CSV
all_seed_global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/random/kondrak_global_summary_all_seeds.csv",
    index=False
)


=== Running pca_seed=0 ===

=== Running pca_seed=0, pca_dim=5 ===

=== Running pca_seed=0, pca_dim=142 ===

=== Running pca_seed=0, pca_dim=997 ===

=== Running pca_seed=0, pca_dim=2084 ===

=== Running pca_seed=0, pca_dim=3031 ===

=== Running pca_seed=0, pca_dim=3459 ===

=== Running pca_seed=0, pca_dim=3938 ===

=== Running pca_seed=0, pca_dim=4088 ===

=== Running pca_seed=0, pca_dim=4096 ===

=== Running pca_seed=42 ===

=== Running pca_seed=42, pca_dim=5 ===

=== Running pca_seed=42, pca_dim=142 ===

=== Running pca_seed=42, pca_dim=997 ===

=== Running pca_seed=42, pca_dim=2084 ===

=== Running pca_seed=42, pca_dim=3031 ===

=== Running pca_seed=42, pca_dim=3459 ===

=== Running pca_seed=42, pca_dim=3938 ===

=== Running pca_seed=42, pca_dim=4088 ===

=== Running pca_seed=42, pca_dim=4096 ===

=== Running pca_seed=1000 ===

=== Running pca_seed=1000, pca_dim=5 ===

=== Running pca_seed=1000, pca_dim=142 ===

=== Running pca_seed=1000, pca_dim=997 ===

=== Running pca_seed=1000,

# Mistral-8x7B

In [3]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "mistralai/Mixtral-8x7B-v0.1"
space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [8, 158, 1111, 2156, 3052, 3457, 3957, 4093, 4096]

[✓] Tokenizer loaded from: mistralai/Mixtral-8x7B-v0.1/tokenizer


In [6]:
# Collect global statistics for each PCA dimension
global_rows = []

for pca_dim in pca_dim_list:
    print(f"\n=== Running pca_dim={pca_dim} ===")

    # Construct output path for cluster-level CSV
    cluster_outpath = (
        f"{out_root}/{model_name}/{space_name}/morph/"
        f"kondrak_clusters_pca_{pca_dim}.csv"
    )

    Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)
    # Run morphology analysis for the current partition
    # Returns:
    #   - cluster_df: per-cluster metrics
    #   - global_row: aggregated statistics for this PCA dimension
    cluster_df, global_row = run_kondrak_morphology_for_partition(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_dim=pca_dim,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        token_col=token_col,
        cluster_id_col=cluster_id_col,
        ddof=0,
        print_columns=False,
        save_cluster_csv=True,               # enable saving cluster-level CSV
        tokenizer=tokenizer,
        cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
    )

    # Store global summary for this PCA dimension
    global_rows.append(global_row)

# Concatenate all global summaries into a single DataFrame
global_summary_df = concat_global_summary_rows(global_rows)

# Sort by PCA dimension for consistent ordering
global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

# Save global summary CSV
global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/kondrak_global_summary.csv",
    index=False
)


=== Running pca_dim=8 ===

=== Running pca_dim=158 ===

=== Running pca_dim=1111 ===

=== Running pca_dim=2156 ===

=== Running pca_dim=3052 ===

=== Running pca_dim=3457 ===

=== Running pca_dim=3957 ===

=== Running pca_dim=4093 ===

=== Running pca_dim=4096 ===


In [7]:
# =========================================================
# sum
# =========================================================
global_summary_df = concat_global_summary_rows(global_rows)

global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

global_summary_df

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,mistralai/Mixtral-8x7B-v0.1,output_proj,8,True,5,5,euclidean,eom,0.0,0.163004,0.152498,87
1,mistralai/Mixtral-8x7B-v0.1,output_proj,158,True,5,5,euclidean,eom,0.0,0.493500,0.201455,228
2,mistralai/Mixtral-8x7B-v0.1,output_proj,1111,True,5,5,euclidean,eom,0.0,0.532820,0.138891,978
3,mistralai/Mixtral-8x7B-v0.1,output_proj,2156,True,5,5,euclidean,eom,0.0,0.534623,0.136081,1082
4,mistralai/Mixtral-8x7B-v0.1,output_proj,3052,True,5,5,euclidean,eom,0.0,0.527606,0.137346,1098
5,mistralai/Mixtral-8x7B-v0.1,output_proj,3457,True,5,5,euclidean,eom,0.0,0.523427,0.138402,1097
6,mistralai/Mixtral-8x7B-v0.1,output_proj,3957,True,5,5,euclidean,eom,0.0,0.528498,0.138166,1094
7,mistralai/Mixtral-8x7B-v0.1,output_proj,4093,True,5,5,euclidean,eom,0.0,0.528488,0.138452,1091
8,mistralai/Mixtral-8x7B-v0.1,output_proj,4096,True,5,5,euclidean,eom,0.0,0.528300,0.138757,1091


### random

In [4]:
from pathlib import Path

# Collect global statistics for each seed and PCA dimension
all_seed_global_summary = []

for pca_seed in seed_list:
    print(f"\n==============================")
    print(f"=== Running pca_seed={pca_seed} ===")
    print(f"==============================")

    global_rows = []

    for pca_dim in pca_dim_list:
        print(f"\n=== Running pca_seed={pca_seed}, pca_dim={pca_dim} ===")

        # Construct output path for cluster-level CSV
        cluster_outpath = (
            f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/morph/"
            f"kondrak_clusters_pca_{pca_dim}.csv"
        )

        Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)

        # Run morphology analysis for the current random-PCA partition
        # Returns:
        #   - cluster_df: per-cluster metrics
        #   - global_row: aggregated statistics for this PCA dimension
        cluster_df, global_row = run_kondrak_morphology_for_random_pca_partition(
            out_root=out_root,
            model_name=model_name,
            space_name=space_name,
            pca_seed=pca_seed,
            pca_dim=pca_dim,
            l2_norm=l2_norm,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=metric,
            cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon,
            token_col=token_col,
            cluster_id_col=cluster_id_col,
            ddof=0,
            print_columns=False,
            save_cluster_csv=True,               # enable saving cluster-level CSV
            tokenizer=tokenizer,
            cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
        )

        # Store global summary for this PCA dimension
        global_rows.append(global_row)

    # Concatenate all global summaries for this seed into a single DataFrame
    global_summary_df = concat_global_summary_rows(global_rows)

    # Sort by PCA dimension for consistent ordering
    global_summary_df.sort_values("pca_dim", inplace=True)
    global_summary_df.reset_index(drop=True, inplace=True)

    # Save per-seed global summary CSV
    seed_summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/"
        f"kondrak_global_summary.csv"
    )
    Path(seed_summary_outpath).parent.mkdir(parents=True, exist_ok=True)
    global_summary_df.to_csv(seed_summary_outpath, index=False)

    all_seed_global_summary.append(global_summary_df)

# Optional: concatenate all seeds into one summary DataFrame
all_seed_global_summary_df = concat_global_summary_rows(all_seed_global_summary)

# Sort for consistent ordering
all_seed_global_summary_df.sort_values(
    ["pca_seed", "pca_dim"], inplace=True
)
all_seed_global_summary_df.reset_index(drop=True, inplace=True)

# Save all-seed summary CSV
all_seed_global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/random/kondrak_global_summary_all_seeds.csv",
    index=False
)


=== Running pca_seed=0 ===

=== Running pca_seed=0, pca_dim=8 ===

=== Running pca_seed=0, pca_dim=158 ===

=== Running pca_seed=0, pca_dim=1111 ===

=== Running pca_seed=0, pca_dim=2156 ===

=== Running pca_seed=0, pca_dim=3052 ===

=== Running pca_seed=0, pca_dim=3457 ===

=== Running pca_seed=0, pca_dim=3957 ===

=== Running pca_seed=0, pca_dim=4093 ===

=== Running pca_seed=0, pca_dim=4096 ===

=== Running pca_seed=42 ===

=== Running pca_seed=42, pca_dim=8 ===

=== Running pca_seed=42, pca_dim=158 ===

=== Running pca_seed=42, pca_dim=1111 ===

=== Running pca_seed=42, pca_dim=2156 ===

=== Running pca_seed=42, pca_dim=3052 ===

=== Running pca_seed=42, pca_dim=3457 ===

=== Running pca_seed=42, pca_dim=3957 ===

=== Running pca_seed=42, pca_dim=4093 ===

=== Running pca_seed=42, pca_dim=4096 ===

=== Running pca_seed=1000 ===

=== Running pca_seed=1000, pca_dim=8 ===

=== Running pca_seed=1000, pca_dim=158 ===

=== Running pca_seed=1000, pca_dim=1111 ===

=== Running pca_seed=10

# gpt-oss-20B

In [4]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "gpt-oss"
space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [6, 182, 466, 739, 1591, 2264, 2532, 2868, 2880]

[✓] Tokenizer loaded from: gpt-oss/tokenizer


In [5]:
# Collect global statistics for each PCA dimension
global_rows = []

for pca_dim in pca_dim_list:
    print(f"\n=== Running pca_dim={pca_dim} ===")

    # Construct output path for cluster-level CSV
    cluster_outpath = (
        f"{out_root}/{model_name}/{space_name}/morph/"
        f"kondrak_clusters_pca_{pca_dim}.csv"
    )

    Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)
    # Run morphology analysis for the current partition
    # Returns:
    #   - cluster_df: per-cluster metrics
    #   - global_row: aggregated statistics for this PCA dimension
    cluster_df, global_row = run_kondrak_morphology_for_partition(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_dim=pca_dim,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        token_col=token_col,
        cluster_id_col=cluster_id_col,
        ddof=0,
        print_columns=False,
        save_cluster_csv=True,               # enable saving cluster-level CSV
        tokenizer=tokenizer,
        cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
    )

    # Store global summary for this PCA dimension
    global_rows.append(global_row)

# Concatenate all global summaries into a single DataFrame
global_summary_df = concat_global_summary_rows(global_rows)

# Sort by PCA dimension for consistent ordering
global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

# Save global summary CSV
global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/kondrak_global_summary.csv",
    index=False
)


=== Running pca_dim=6 ===

=== Running pca_dim=182 ===

=== Running pca_dim=466 ===

=== Running pca_dim=739 ===

=== Running pca_dim=1591 ===

=== Running pca_dim=2264 ===

=== Running pca_dim=2532 ===

=== Running pca_dim=2868 ===

=== Running pca_dim=2880 ===


In [6]:
# =========================================================
# sum
# =========================================================
global_summary_df = concat_global_summary_rows(global_rows)

global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

global_summary_df

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,gpt-oss,output_proj,6,True,5,5,euclidean,eom,0.0,0.229117,0.153079,1203
1,gpt-oss,output_proj,182,True,5,5,euclidean,eom,0.0,0.456162,0.198569,740
2,gpt-oss,output_proj,466,True,5,5,euclidean,eom,0.0,0.547680,0.182162,1941
3,gpt-oss,output_proj,739,True,5,5,euclidean,eom,0.0,0.554638,0.179798,2783
4,gpt-oss,output_proj,1591,True,5,5,euclidean,eom,0.0,0.543603,0.174769,3625
5,gpt-oss,output_proj,2264,True,5,5,euclidean,eom,0.0,0.539204,0.177650,3819
6,gpt-oss,output_proj,2532,True,5,5,euclidean,eom,0.0,0.537162,0.179507,3856
7,gpt-oss,output_proj,2868,True,5,5,euclidean,eom,0.0,0.533994,0.179293,3880
8,gpt-oss,output_proj,2880,True,5,5,euclidean,eom,0.0,0.533820,0.179200,3880


### random

In [7]:
from pathlib import Path

# Collect global statistics for each seed and PCA dimension
all_seed_global_summary = []

for pca_seed in seed_list:
    print(f"\n==============================")
    print(f"=== Running pca_seed={pca_seed} ===")
    print(f"==============================")

    global_rows = []

    for pca_dim in pca_dim_list:
        print(f"\n=== Running pca_seed={pca_seed}, pca_dim={pca_dim} ===")

        # Construct output path for cluster-level CSV
        cluster_outpath = (
            f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/morph/"
            f"kondrak_clusters_pca_{pca_dim}.csv"
        )

        Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)

        # Run morphology analysis for the current random-PCA partition
        # Returns:
        #   - cluster_df: per-cluster metrics
        #   - global_row: aggregated statistics for this PCA dimension
        cluster_df, global_row = run_kondrak_morphology_for_random_pca_partition(
            out_root=out_root,
            model_name=model_name,
            space_name=space_name,
            pca_seed=pca_seed,
            pca_dim=pca_dim,
            l2_norm=l2_norm,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=metric,
            cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon,
            token_col=token_col,
            cluster_id_col=cluster_id_col,
            ddof=0,
            print_columns=False,
            save_cluster_csv=True,               # enable saving cluster-level CSV
            tokenizer=tokenizer,
            cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
        )

        # Store global summary for this PCA dimension
        global_rows.append(global_row)

    # Concatenate all global summaries for this seed into a single DataFrame
    global_summary_df = concat_global_summary_rows(global_rows)

    # Sort by PCA dimension for consistent ordering
    global_summary_df.sort_values("pca_dim", inplace=True)
    global_summary_df.reset_index(drop=True, inplace=True)

    # Save per-seed global summary CSV
    seed_summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/"
        f"kondrak_global_summary.csv"
    )
    Path(seed_summary_outpath).parent.mkdir(parents=True, exist_ok=True)
    global_summary_df.to_csv(seed_summary_outpath, index=False)

    all_seed_global_summary.append(global_summary_df)

# Optional: concatenate all seeds into one summary DataFrame
all_seed_global_summary_df = concat_global_summary_rows(all_seed_global_summary)

# Sort for consistent ordering
all_seed_global_summary_df.sort_values(
    ["pca_seed", "pca_dim"], inplace=True
)
all_seed_global_summary_df.reset_index(drop=True, inplace=True)

# Save all-seed summary CSV
all_seed_global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/random/kondrak_global_summary_all_seeds.csv",
    index=False
)


=== Running pca_seed=0 ===

=== Running pca_seed=0, pca_dim=6 ===

=== Running pca_seed=0, pca_dim=182 ===

=== Running pca_seed=0, pca_dim=466 ===

=== Running pca_seed=0, pca_dim=739 ===

=== Running pca_seed=0, pca_dim=1591 ===

=== Running pca_seed=0, pca_dim=2264 ===

=== Running pca_seed=0, pca_dim=2532 ===

=== Running pca_seed=0, pca_dim=2868 ===

=== Running pca_seed=0, pca_dim=2880 ===

=== Running pca_seed=42 ===

=== Running pca_seed=42, pca_dim=6 ===

=== Running pca_seed=42, pca_dim=182 ===

=== Running pca_seed=42, pca_dim=466 ===

=== Running pca_seed=42, pca_dim=739 ===

=== Running pca_seed=42, pca_dim=1591 ===

=== Running pca_seed=42, pca_dim=2264 ===

=== Running pca_seed=42, pca_dim=2532 ===

=== Running pca_seed=42, pca_dim=2868 ===

=== Running pca_seed=42, pca_dim=2880 ===

=== Running pca_seed=1000 ===

=== Running pca_seed=1000, pca_dim=6 ===

=== Running pca_seed=1000, pca_dim=182 ===

=== Running pca_seed=1000, pca_dim=466 ===

=== Running pca_seed=1000, p